In [10]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import time
import random

In [11]:
BASE_URL = 'https://auto.ria.com'

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
}

session = requests.Session()
session.headers.update(HEADERS)


def get_text(tag):
    return tag.text.strip() if tag else None


res_list = []
failed_pages = []

for i in range(0, 50):
    url = f'{BASE_URL}/uk/search/?categories.main.id=1&limit=100&page={i}'

    try:
        page = session.get(url, timeout=10)
    except requests.exceptions.RequestException as e:
        print(f'Сторінка {i}: помилка з\'єднання — {e}, пропускаю')
        failed_pages.append(i)
        time.sleep(5)
        continue

    if page.status_code != 200:
        print(f'Сторінка {i}: статус {page.status_code}, пропускаю')
        failed_pages.append(i)
        time.sleep(5)
        continue

    soup = BeautifulSoup(page.text, 'lxml')
    cars = soup.find_all('a', class_='product-card')

    if not cars:
        print(f'Сторінка {i}: 0 карток знайдено — можлива капча або зміна розмітки')
        failed_pages.append(i)

    for car in cars:
        car_info = {}

        name_tag = car.find('div', class_='common-text size-16-20 titleS fw-bold mb-4')
        price_tag = car.find('span', class_='common-text titleM c-green')
        car_info['name'] = get_text(name_tag)
        car_info['price'] = get_text(price_tag)

        rows = car.find_all('div', class_='structure-row ai-center gap-8 flex-1')
        mileage = get_text(rows[0].find('span', class_='common-text ellipsis-1 body')) if len(rows) > 0 else None
        idx1_text = get_text(rows[1].find('span', class_='common-text ellipsis-1 body')) if len(rows) > 1 else None
        fuel = get_text(rows[2].find('span', class_='common-text ellipsis-1 body')) if len(rows) > 2 else None
        city = get_text(rows[3].find('span', class_='common-text ellipsis-1 body')) if len(rows) > 3 else None

        if fuel and 'Електро' in fuel:
            transmission, range_km = None, idx1_text
        else:
            transmission, range_km = idx1_text, None

        car_info['mileage'] = mileage
        car_info['transmission'] = transmission
        car_info['range_km'] = range_km
        car_info['fuel'] = fuel
        car_info['city'] = city

        badges = [get_text(b) for b in car.find_all('span', class_='common-text ws-pre-wrap badge')]
        car_info['road_accident'] = 'Був у ДТП' if any(b and 'ДТП' in b for b in badges) else 'Не був у ДТП'

        href = car.get('href')
        car_info['link'] = BASE_URL + href if href else None

        res_list.append(car_info)

    time.sleep(random.uniform(1.5, 3.5))  # пауза ПІСЛЯ кожної сторінки, перед наступним запитом

print(f'Зібрано {len(res_list)} рядків. Проблемних сторінок: {failed_pages}')

Зібрано 4993 рядків. Проблемних сторінок: []


In [12]:
df = pd.DataFrame(res_list)
df.to_csv('../data/auto_ria_raw.csv', index=False, encoding='utf-8-sig')
df

,name,price,mileage,transmission,range_km,fuel,city,road_accident,link
0,Haval Dargo 2022,23 800 $,18 тис. км,Автомат,NaN,"Бензин, 2 л",Сміла,Не був у ДТП,https://auto.ria.com/uk/auto_haval_dargo_39547...
1,BMW 5 Series 2017,29 999 $,133 тис. км,Автомат,NaN,"Бензин, 3 л",Львів (Львівська),Був у ДТП,https://auto.ria.com/uk/auto_bmw_5-series_3722...
2,Maxus e-Deliver 3 2022,25 800 $,12 тис. км,NaN,225 км,"Електро, 50 кВт·год",Київ,Не був у ДТП,https://auto.ria.com/uk/auto_maxus_e-deliver-3...
3,BMW 5 Series 2017,31 500 $,127 тис. км,Автомат,NaN,"Дизель, 2 л",Одеса,Не був у ДТП,https://auto.ria.com/uk/auto_bmw_5-series_4003...
4,Renault Talisman 2018,13 900 $,231 тис. км,Автомат,NaN,"Дизель, 1.5 л",Хмельницький,Не був у ДТП,https://auto.ria.com/uk/auto_renault_talisman_...
...,...,...,...,...,...,...,...,...,...
4988,Volkswagen Touran 2007,7 290 $,260 тис. км,Автомат,NaN,"Дизель, 2 л",Вінниця,Не був у ДТП,https://auto.ria.com/uk/auto_volkswagen_touran...
4989,Renault Master 2015,13 500 $,470 тис. км,Ручна / Механіка,NaN,"Дизель, 2.3 л",Київ,Не був у ДТП,https://auto.ria.com/uk/auto_renault_master_40...
4990,ЗАЗ Sens 2007,1 550 $,119 тис. км,Ручна / Механіка,NaN,"Бензин, 1.3 л",Київ,Не був у ДТП,https://auto.ria.com/uk/auto_zaz_sens_39955310...
4991,Ford Mondeo 2012,6 500 $,338 тис. км,Ручна / Механіка,NaN,"Дизель, 1.6 л",Київ,Не був у ДТП,https://auto.ria.com/uk/auto_ford_mondeo_36872...
